In [1]:
cell_str='''
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <time.h>
#include <cuda_runtime.h>

// --- KERNEL NAIVE ---
// Nessuna ottimizzazione, nessuna memoria condivisa.
// Ogni thread legge direttamente dalla lentissima Memoria Globale.
__global__ void matMulNaive(const double *a, const double *b, double *c, int n) {
    // I cicli "i" e "j" scompaiono: le coordinate sono calcolate tramite gli indici dei thread
    int row = blockIdx.y * blockDim.y + threadIdx.y; // Sostituisce la 'i'
    int col = blockIdx.x * blockDim.x + threadIdx.x; // Sostituisce la 'j'

    // Controllo dei bordi: ci assicuriamo che il thread sia dentro la matrice
    if (row < n && col < n) {
        double sum = 0.0;

        // Questo è l'unico ciclo che sopravvive (corrisponde al ciclo 'k' originale)
        for (int k = 0; k < n; ++k) {
            // Lettura pesantissima direttamente dalla memoria globale (VRAM)
            sum += a[row * n + k] * b[k * n + col];
        }

        c[row * n + col] = sum;
    }
}

int main(int argc, char **argv) {
    if (argc < 2) {
        fprintf(stderr, "Error: missing n argument.\\n");
        fprintf(stderr, "Usage: %s <n>\\n", argv[0]);
        return 1;
    }

    int n = atoi(argv[1]);
    if (n <= 0) {
        fprintf(stderr, "Error: You forgot to provide n!\\n");
        return 1;
    }

    // Calcoliamo i byte necessari (usando double: 8 byte per elemento)
    size_t bytes = (size_t)n * n * sizeof(double);

    // Allocazione Host (CPU)
    // NOTA: In CUDA si preferisce sempre "appiattire" le matrici in array 1D
    double *h_a = (double*)malloc(bytes);
    double *h_b = (double*)malloc(bytes);
    double *h_c = (double*)malloc(bytes);

    if (!h_a || !h_b || !h_c) {
        fprintf(stderr, "Errore di allocazione RAM Host!\\n");
        return 1;
    }

    // Inizializzazione
    for (int i = 0; i < n; i++) {
        for (int j = 0; j < n; j++) {
            h_a[i * n + j] = 2.0;
            h_b[i * n + j] = 3.0;
            h_c[i * n + j] = 0.0;
        }
    }

    // Allocazione Device (GPU)
    double *d_a, *d_b, *d_c;
    cudaMalloc(&d_a, bytes);
    cudaMalloc(&d_b, bytes);
    cudaMalloc(&d_c, bytes);

    // Trasferimento dati HtoD
    cudaMemcpy(d_a, h_a, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, bytes, cudaMemcpyHostToDevice);

    // Configurazione della Griglia di esecuzione (Blocchi 32x32 = 1024 thread)
    int blockSize = 32;
    dim3 threadsPerBlock(blockSize, blockSize);
    dim3 numBlocks((n + blockSize - 1) / blockSize, (n + blockSize - 1) / blockSize);

    printf("Starting the computation (CUDA Naive - FP64)...\\n");
    clock_t start = clock();

    // Lancio del kernel
    matMulNaive<<<numBlocks, threadsPerBlock>>>(d_a, d_b, d_c, n);

    // La CPU deve aspettare che la GPU finisca prima di fermare il cronometro
    cudaDeviceSynchronize();
    clock_t end = clock();

    double duration = (double)(end - start) / CLOCKS_PER_SEC;
    printf("Execution Time: %.4f seconds\\n", duration);

    // Trasferimento risultati DtoH
    cudaMemcpy(h_c, d_c, bytes, cudaMemcpyDeviceToHost);

    // Salvataggio dei risultati (con salvaguardia per n piccoli)
    FILE *f = fopen("mat-res.txt", "w");
    if (!f) {
        perror("fopen");
        return 1;
    }

    fprintf(f, "%d\\n\\n", n);
    int sample_size = (n < 1000) ? n : 1000;
    for (int i = 0; i < sample_size; i++) {
        for (int j = 0; j < sample_size; j++) {
            fprintf(f, "%.0f ", h_c[i * n + j]);
        }
        fprintf(f, "\\n");
    }

    fclose(f);

    // Pulizia finale
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);
    free(h_a);
    free(h_b);
    free(h_c);

    return 0;
}
'''

with open('cuda_matrixmult_naive.cu', 'w') as f:
    f.write(cell_str)

In [2]:
!nvcc -arch=sm_75 -O3 cuda_matrixmult_naive.cu -o cuda_matrixmult_naive
!nvprof ./cuda_matrixmult_naive 20000

==1032== NVPROF is profiling process 1032, command: ./cuda_matrixmult_naive 20000
Starting the computation (CUDA Naive - FP64)...
Execution Time: 95.7564 seconds
==1032== Profiling application: ./cuda_matrixmult_naive 20000
==1032== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   97.90%  97.8282s         1  97.8282s  97.8282s  97.8282s  matMulNaive(double const *, double const *, double*, int)
                    1.35%  1.35220s         2  676.10ms  670.28ms  681.92ms  [CUDA memcpy HtoD]
                    0.74%  743.74ms         1  743.74ms  743.74ms  743.74ms  [CUDA memcpy DtoH]
      API calls:   97.69%  97.8283s         1  97.8283s  97.8283s  97.8283s  cudaDeviceSynchronize
                    2.10%  2.09798s         3  699.33ms  670.50ms  744.27ms  cudaMemcpy
                    0.20%  202.75ms         3  67.582ms  203.38us  202.31ms  cudaMalloc
                    0.01%  8.1327ms         3  2.7109ms  1.5641ms